# BIG WIP

In [36]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString
from shapely import wkt
from shapely.ops import unary_union, linemerge, nearest_points
import plotly.express as px
from helper_functions import quick_px_scattermap

In [2]:
station_avg_delay = pd.read_csv("../data/processed/cleaned_delay_data.csv")
station_avg_ridership = pd.read_csv("../data/processed/selected_station_ridership.csv")
selected_routes_shapes = pd.read_csv("../data/processed/selected_routes.csv")
sj_adt = pd.read_csv("../data/processed/selected_sj_adt.csv")

selected_routes = ['22', 'Rapid 522', '23', 'Rapid 523', '25', '60']
station_avg_delay = station_avg_delay[station_avg_delay["route_id"].isin(selected_routes)]
station_avg_delay = station_avg_delay.groupby(['route_id', 'direction_id', 'stop_id', 'stop_sequence']).agg(mean_delay=("computed_delay_sec", "mean")).reset_index().sort_values(['route_id', 'direction_id', 'stop_sequence'])
station_avg_delay["route_id"] = station_avg_delay["route_id"].str.replace(
    r"^Rapid\s+", "", regex=True
).astype(int)
station_avg_delay

,route_id,direction_id,stop_id,stop_sequence,mean_delay
18,22,0,60328,2,139.500000
19,22,0,60329,3,196.000000
20,22,0,60330,4,173.000000
21,22,0,60331,5,146.500000
22,22,0,60332,6,158.666667
...,...,...,...,...,...
586,523,1,60651,13,-466.000000
587,523,1,60653,14,-488.500000
588,523,1,60656,15,-525.000000
589,523,1,60660,16,-604.500000


In [3]:
station_avg_ridership['direction_id'] = station_avg_ridership['direction_id'] % 10
station_avg_ridership = station_avg_ridership.sort_values(['route_id', 'direction_id', 'stop_id'])
station_avg_ridership

,route_id,direction_id,stop_id,stop_name,boardings,alightings,total_b_a,geometry
0,22,0,60001,Santa Clara Transit Center,152.880022,110.852250,263.732272,POINT (6144099.62758 1954225.5200159)
4,22,0,60020,El Camino & Lafayette,40.890986,57.438087,98.329073,POINT (6141651.00288408 1954992.24617273)
6,22,0,60035,King & Alum Rock,149.607974,157.936236,307.544210,POINT (6167727.93202934 1953654.17929506)
7,22,0,60036,King & San Antonio,36.418456,38.469837,74.888293,POINT (6168552.89461824 1952565.84026439)
8,22,0,60037,King & Hermocilla,31.495157,45.198273,76.693430,POINT (6169311.51383591 1951509.07203673)
...,...,...,...,...,...,...,...,...
586,523,1,64478,Lockheed Martin Transit Center,0.000000,317.466630,317.466630,POINT (6117801.90097608 1975568.38743506)
601,523,1,64696,Stevens Creek & Winchester,85.577443,90.755989,176.333432,POINT (6139677.16759291 1943518.2914304)
620,523,1,65741,Stevens Creek & Valley Fair / Santana Row,109.060532,391.806776,500.867308,POINT (6141126.594156 1943538.09191573)
642,523,1,65902,Sunnyvale-Saratoga & El Camino,189.107898,219.736022,408.843920,POINT (6116423.22131875 1959757.18710214)


In [4]:
def safe_wkt_load(x):
    if pd.isna(x):
        return None
    return wkt.loads(x)

stations_df = pd.merge(station_avg_delay, station_avg_ridership, on=['route_id', 'direction_id', 'stop_id'], how='left')
stations_df["geometry"] = stations_df["geometry"].apply(safe_wkt_load)
stations_gdf = gpd.GeoDataFrame(stations_df, geometry='geometry', crs="EPSG:2227").sort_values(["route_id", "direction_id", "stop_sequence"])
stations_gdf

,route_id,direction_id,stop_id,stop_sequence,mean_delay,stop_name,boardings,alightings,total_b_a,geometry
0,22,0,60328,2,139.500000,Palo Alto Transit Center (Bay 10),733.994859,0.238095,734.176677,POINT (6078107.96 1988346.454)
1,22,0,60329,3,196.000000,El Camino & Palm,12.793264,3.426027,16.219290,POINT (6079055.872 1986640.972)
2,22,0,60330,4,173.000000,El Camino & Galvez,66.459791,7.428228,73.888019,POINT (6079788.226 1985807.876)
3,22,0,60331,5,146.500000,El Camino & Sam McDonald,4.579660,0.262500,4.842160,POINT (6080518.117 1984980.671)
4,22,0,60332,6,158.666667,El Camino & Churchill,4.251791,1.055649,5.307440,POINT (6081284.46 1984093.848)
...,...,...,...,...,...,...,...,...,...,...
590,523,1,60651,13,-466.000000,Stevens Creek & Cabot,52.817891,62.395022,115.212913,POINT (6127904.459 1943613.843)
591,523,1,60653,14,-488.500000,NaN,NaN,NaN,NaN,None
592,523,1,60656,15,-525.000000,Stevens Creek & Wolfe,66.990445,88.557801,155.548246,POINT (6121122.696 1943673.642)
593,523,1,60660,16,-604.500000,Stevens Creek & De Anza,85.345256,218.676593,304.021849,POINT (6115865.544 1943743.247)


In [5]:
selected_routes_shapes["geometry"] = selected_routes_shapes["geometry"].apply(safe_wkt_load)
selected_routes_shapes_gdf = gpd.GeoDataFrame(selected_routes_shapes, geometry='geometry', crs="EPSG:2227")
selected_routes_shapes_gdf

,line_id,route_id,route_desc,category,route_length,object_id,geometry
0,23875,22,PALO ALTO - EASTRIDGE,Frequent,242559.907697,3,"MULTILINESTRING ((6078120.21 1988261.911, 6078..."
1,23876,23,DE ANZA COL - ALUM ROCK STN,Frequent,120202.952188,4,"MULTILINESTRING ((6158970.5 1948773.359, 61591..."
2,23879,25,DE ANZA COL - VMC - ALUM ROCK STN,Frequent,140850.289491,5,"MULTILINESTRING ((6159010.974 1938537.442, 615..."
3,23904,60,WINCHESTER STN - MILPITAS BART,Frequent,151256.059019,21,"MULTILINESTRING ((6140271.837 1927485.307, 614..."
4,23896,522,PALO ALTO - EASTRIDGE,Rapid,253522.042435,55,"MULTILINESTRING ((6078120.21 1988261.911, 6078..."
5,23897,523,SAN JOSE STATE - LOCKHEED MARTIN,Rapid,162676.690292,56,"MULTILINESTRING ((6117718.556 1975678.574, 611..."


In [39]:
def compute_station_spacing_by_route(stations_gdf, routes_gdf):
    results = []
    stations_gdf = stations_gdf.to_crs(routes_gdf.crs)

    for (route_id, direction_id), stops in stations_gdf.groupby(["route_id", "direction_id"]):
        shape_row = routes_gdf[routes_gdf["route_id"] == route_id]
        if shape_row.empty:
            continue

        #merge MultiLineStrings into single shape if possible
        geom_list = shape_row.geometry.tolist()
        merged = unary_union(geom_list)
        line = linemerge(merged)

        #fallback
        if line.geom_type == "MultiLineString":
            line = max(line.geoms, key=lambda g: g.length)

        # projects stops onto route and calculate spacings
        stops = stops.sort_values("stop_sequence").copy()
        stops["proj_dist"] = stops.geometry.apply(lambda p: line.project(p))
        stops["spacing"] = stops["proj_dist"].diff()

        results.append(stops)

    if not results:
        raise ValueError("No matching route_id groups found between stations and routes")

    return pd.concat(results, ignore_index=True)

In [40]:
station_spacings = compute_station_spacing_by_route(stations_gdf, selected_routes_shapes_gdf)
station_spacings

,route_id,direction_id,stop_id,stop_sequence,mean_delay,stop_name,boardings,alightings,total_b_a,geometry,proj_dist,spacing
0,22,0,60328,2,139.500000,Palo Alto Transit Center (Bay 10),733.994859,0.238095,734.176677,POINT (6078107.96 1988346.454),0.000000,NaN
1,22,0,60329,3,196.000000,El Camino & Palm,12.793264,3.426027,16.219290,POINT (6079055.872 1986640.972),1912.483986,1912.483986
2,22,0,60330,4,173.000000,El Camino & Galvez,66.459791,7.428228,73.888019,POINT (6079788.226 1985807.876),3020.952373,1108.468387
3,22,0,60331,5,146.500000,El Camino & Sam McDonald,4.579660,0.262500,4.842160,POINT (6080518.117 1984980.671),4125.164662,1104.212289
4,22,0,60332,6,158.666667,El Camino & Churchill,4.251791,1.055649,5.307440,POINT (6081284.46 1984093.848),5297.244257,1172.079595
...,...,...,...,...,...,...,...,...,...,...,...,...
590,523,1,60651,13,-466.000000,Stevens Creek & Cabot,52.817891,62.395022,115.212913,POINT (6127904.459 1943613.843),0.000000,0.000000
591,523,1,60653,14,-488.500000,NaN,NaN,NaN,NaN,None,NaN,NaN
592,523,1,60656,15,-525.000000,Stevens Creek & Wolfe,66.990445,88.557801,155.548246,POINT (6121122.696 1943673.642),0.000000,NaN
593,523,1,60660,16,-604.500000,Stevens Creek & De Anza,85.345256,218.676593,304.021849,POINT (6115865.544 1943743.247),0.000000,0.000000


In [29]:
station_spacings.to_csv("../data/processed/temp.csv")